# 15 Enzyme Kinetics: Michaelis-Menten-style Curve - Python

## Biochemistry question

How does initial reaction velocity change as substrate concentration increases in a synthetic enzyme kinetics dataset?

This notebook introduces Michaelis-Menten-style thinking with synthetic data. The fitted values are educational estimates only, not real enzyme parameters.


In [ ]:
import plotly.io as pio
pio.renderers.default = "iframe"


## 1. Setup

We use `pandas` for data tables, `plotly` for visualization, and small helper functions from `src/enzyme_kinetics.py`.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

PROJECT_ROOT = Path("..").resolve()
sys.path.append(str(PROJECT_ROOT))

from src.enzyme_kinetics import fit_michaelis_menten, make_prediction_table, summarize_velocity


## 2. Dataset Preview

Each row is a synthetic replicate measurement at one substrate concentration. The response is `initial_velocity`, measured in arbitrary teaching units per minute.


In [ ]:
clean_path = PROJECT_ROOT / "data" / "enzyme_kinetics" / "michaelis_menten_clean.csv"
noisy_path = PROJECT_ROOT / "data" / "enzyme_kinetics" / "michaelis_menten_noisy.csv"

clean_df = pd.read_csv(clean_path)
noisy_df = pd.read_csv(noisy_path)
clean_df.head()


## 3. Summary Statistics

Summarize the replicate velocities at each substrate concentration. This helps separate the average pattern from replicate-level variation.


In [ ]:
clean_summary = summarize_velocity(clean_df)
clean_summary


## 4. Plot the Clean Dataset

Michaelis-Menten-style data often show a saturation pattern: velocity increases with substrate concentration and then begins to level off.


In [ ]:
fig = px.scatter(
    clean_summary,
    x="substrate_mM",
    y="mean_velocity",
    error_y="sem_velocity",
    title="Synthetic Enzyme Kinetics: Mean Initial Velocity",
    labels={"substrate_mM": "Substrate (mM)", "mean_velocity": "Mean Initial Velocity"},
)
fig.update_xaxes(title="Substrate concentration (mM)")
fig.update_yaxes(title="Mean initial velocity")
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


## 5. Fit an Educational Michaelis-Menten Curve

The helper function fits a simple curve of the form:

```text
velocity = (Vmax * substrate) / (Km + substrate)
```

For this notebook, `Vmax` and `Km` are teaching estimates. They should be interpreted cautiously.


In [ ]:
clean_params = fit_michaelis_menten(clean_summary)
clean_curve = make_prediction_table(
    clean_params,
    min_substrate=clean_summary["substrate_mM"].min(),
    max_substrate=clean_summary["substrate_mM"].max(),
)

{"educational_vmax": round(clean_params["vmax"], 3), "educational_km": round(clean_params["km"], 3)}


In [ ]:
fig = px.scatter(
    clean_summary,
    x="substrate_mM",
    y="mean_velocity",
    error_y="sem_velocity",
    title="Synthetic Michaelis-Menten-style Fit",
    labels={"substrate_mM": "Substrate (mM)", "mean_velocity": "Mean Initial Velocity"},
)
fig.add_trace(
    go.Scatter(
        x=clean_curve["substrate_mM"],
        y=clean_curve["predicted_velocity"],
        mode="lines",
        name="Fitted curve",
    )
)
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


## 6. Compare Clean and Noisy Data

A noisier dataset can still show the same general saturation pattern, but the fitted curve and interpretation should be treated with more caution.


In [ ]:
noisy_summary = summarize_velocity(noisy_df)
noisy_params = fit_michaelis_menten(noisy_summary)
noisy_curve = make_prediction_table(
    noisy_params,
    min_substrate=noisy_summary["substrate_mM"].min(),
    max_substrate=noisy_summary["substrate_mM"].max(),
)

comparison = pd.DataFrame([
    {"dataset": "clean", "educational_vmax": clean_params["vmax"], "educational_km": clean_params["km"]},
    {"dataset": "noisy", "educational_vmax": noisy_params["vmax"], "educational_km": noisy_params["km"]},
])
comparison.round(3)


In [ ]:
clean_plot = clean_summary.assign(dataset="clean")
noisy_plot = noisy_summary.assign(dataset="noisy")
combined = pd.concat([clean_plot, noisy_plot], ignore_index=True)

fig = px.scatter(
    combined,
    x="substrate_mM",
    y="mean_velocity",
    error_y="sem_velocity",
    color="dataset",
    title="Clean vs Noisy Synthetic Enzyme Kinetics",
    labels={"substrate_mM": "Substrate (mM)", "mean_velocity": "Mean Initial Velocity"},
)
# If this chart does not render in Jupyter, try: fig.show(renderer="browser")
fig.show(renderer="iframe")


## What You Should Notice

- Initial velocity increases as substrate concentration increases.
- The increase begins to level off at higher substrate concentrations.
- The noisy dataset has the same general shape but more replicate variability.
- Fitted parameters are more trustworthy when the data are clean and well-spread across the curve.

## Interpretation Practice

- How would you describe the saturation pattern in one cautious sentence?
- Why are low and high substrate concentrations both useful for curve fitting?
- How does noise affect your confidence in the fitted curve?
- What would you add to the experiment before treating the estimate as reliable?

## Common Mistake

- Do not treat the fitted `Km` or `Vmax` as real enzyme constants. In this notebook they are educational estimates from synthetic data.

## Limitations

- These datasets are synthetic and for learning only.
- The notebook does not check all model assumptions or experimental conditions.
- Real enzyme kinetics would need careful assay timing, substrate range selection, controls, and repeated validation.
- The fitted values should not be used for biological, clinical, diagnostic, or regulatory claims.
